## 2. Missing Values And Coverage

In [3]:
def first_last_non_null_date(frame: pd.DataFrame, column: str) -> tuple[str | None, str | None]:
    mask = frame[column].notna()
    if not mask.any():
        return None, None
    return str(frame.loc[mask, 'quarter_period'].iloc[0]), str(frame.loc[mask, 'quarter_period'].iloc[-1])


def coverage_class(coverage_pct: float) -> str:
    if coverage_pct >= 0.95:
        return 'core_long_sample_candidate'
    if coverage_pct >= 0.80:
        return 'shorter_sample_candidate'
    if coverage_pct >= 0.50:
        return 'limited_sample_candidate'
    return 'exclude_low_coverage'


sample_start = str(df['quarter_period'].min())
coverage_rows = []
for column in df_model.columns:
    if column == 'quarter':
        continue
    first_non_null, last_non_null = first_last_non_null_date(df, column)
    missing_count = int(df[column].isna().sum())
    available_observations = int(df[column].notna().sum())
    missing_pct = missing_count / len(df)
    coverage_pct = available_observations / len(df)
    late_start_reduces_sample = first_non_null not in {None, sample_start}
    substantial_missingness = missing_pct >= MATERIAL_MISSING_THRESHOLD
    coverage_rows.append({
        'variable': column,
        'available_observations': available_observations,
        'non_null_rows': available_observations,
        'missing_count': missing_count,
        'missing_pct': missing_pct,
        'coverage_pct': coverage_pct,
        'start_quarter': sample_start,
        'first_non_null': first_non_null,
        'last_non_null': last_non_null,
        'late_start_reduces_estimation_sample': bool(late_start_reduces_sample),
        'substantial_missingness_warning': bool(substantial_missingness),
        # Backward-compatible name used later in the notebook; this now means
        # genuinely substantial missingness, not merely starting after 1995Q1.
        'material_early_missingness': bool(substantial_missingness),
        'coverage_class': coverage_class(coverage_pct),
    })

coverage = pd.DataFrame(coverage_rows).sort_values(['missing_pct', 'variable'], ascending=[False, True])
display(coverage)
display(coverage.loc[coverage['substantial_missingness_warning'], ['variable', 'missing_pct', 'coverage_pct', 'first_non_null', 'last_non_null', 'coverage_class']])

,variable,available_observations,non_null_rows,missing_count,missing_pct,coverage_pct,start_quarter,first_non_null,last_non_null,late_start_reduces_estimation_sample,substantial_missingness_warning,material_early_missingness,coverage_class
44,household_spending_growth_lag1,52,52,72,0.580645,0.419355,1995Q1,2013Q1,2025Q4,True,True,True,exclude_low_coverage
21,household_spending_growth,53,53,71,0.572581,0.427419,1995Q1,2012Q4,2025Q4,True,True,True,exclude_low_coverage
20,household_spending,54,54,70,0.564516,0.435484,1995Q1,2012Q3,2025Q4,True,True,True,exclude_low_coverage
40,aud_usd_change_lag1,62,62,62,0.500000,0.500000,1995Q1,2010Q3,2025Q4,True,True,True,limited_sample_candidate
19,aud_usd_change,63,63,61,0.491935,0.508065,1995Q1,2010Q2,2025Q4,True,True,True,limited_sample_candidate
18,aud_usd,64,64,60,0.483871,0.516129,1995Q1,2010Q1,2025Q4,True,True,True,limited_sample_candidate
39,brent_growth_lag1,72,72,52,0.419355,0.580645,1995Q1,2008Q1,2025Q4,True,True,True,limited_sample_candidate
17,brent_growth,73,73,51,0.411290,0.588710,1995Q1,2007Q4,2025Q4,True,True,True,limited_sample_candidate
16,brent_price,74,74,50,0.403226,0.596774,1995Q1,2007Q3,2025Q4,True,True,True,limited_sample_candidate
38,wti_growth_lag1,100,100,24,0.193548,0.806452,1995Q1,2001Q1,2025Q4,True,False,False,shorter_sample_candidate


,variable,missing_pct,coverage_pct,first_non_null,last_non_null,coverage_class
44,household_spending_growth_lag1,0.580645,0.419355,2013Q1,2025Q4,exclude_low_coverage
21,household_spending_growth,0.572581,0.427419,2012Q4,2025Q4,exclude_low_coverage
20,household_spending,0.564516,0.435484,2012Q3,2025Q4,exclude_low_coverage
40,aud_usd_change_lag1,0.500000,0.500000,2010Q3,2025Q4,limited_sample_candidate
19,aud_usd_change,0.491935,0.508065,2010Q2,2025Q4,limited_sample_candidate
18,aud_usd,0.483871,0.516129,2010Q1,2025Q4,limited_sample_candidate
39,brent_growth_lag1,0.419355,0.580645,2008Q1,2025Q4,limited_sample_candidate
17,brent_growth,0.411290,0.588710,2007Q4,2025Q4,limited_sample_candidate
16,brent_price,0.403226,0.596774,2007Q3,2025Q4,limited_sample_candidate


**Interpretation.** The worst-coverage predictors — `household_spending_growth`
(43% coverage, doesn't start until 2012Q4), `aud_usd_change` (51%, starts 2010Q2), and
`brent_growth` (59%, starts 2007Q4) — aren't broken data, they're series whose underlying
collection or the project's own retrieval simply doesn't reach back to 1995Q1. The
`coverage_class` column turns this into an explicit modelling decision (`exclude_low_coverage`
vs. `limited_sample_candidate`) rather than a silent one, and it's the reason those same
variables show up later in §6/§7 with much smaller `n_obs` than series like `cpi_qoq` or
`cash_rate` that are complete from the start.